[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihdicaballero/mento/blob/main/docs/source/examples/shear_wall_summary_ACI_318-19%20Design.ipynb)

In [1]:
# Google Colab does not ship mento, so install it when running there.
# Guarded so that building the documentation never shells out to pip.
import sys

if "google.colab" in sys.modules:
    %pip install mento

# Shear Wall Summary Design with ACI 318-19

**Create input dataframe without rebar**

When the mesh columns (dbh, sh, dbv, sv) are left out or zero, `design()` selects the mesh.

In [2]:
from mento import MPa
from mento import ShearWallSummary, Concrete_ACI_318_19, SteelBar
import pandas as pd

conc = Concrete_ACI_318_19(name="H25", f_c=25 * MPa)
steel = SteelBar(name="ADN 420", f_y=420 * MPa)

# One row per wall, keyed by (Level, Label): its geometry.
# The mesh columns are left out: design() selects the mesh.
sections = pd.DataFrame(
    {
        "Level": ["", "Level 1", "Level 2", "Level 1"],
        "Label": ["", "M1", "M1", "M2"],
        "t": ["cm", 20, 20, 20],
        "lw": ["m", 3, 3, 2],
        "hw": ["m", 3, 3, 3],
        "cc": ["mm", 25, 25, 25],
    }
)
# One row per load combination of a wall: Nx > 0 compression, Vz the in-plane shear; My is not used.
forces = pd.DataFrame(
    {
        "Level": [
            "",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 2",
            "Level 2",
            "Level 2",
            "Level 2",
            "Level 1",
            "Level 1",
            "Level 1",
            "Level 1",
        ],
        "Label": ["", "M1", "M1", "M1", "M1", "M1", "M1", "M1", "M1", "M2", "M2", "M2", "M2"],
        "Comb.": [
            "",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
            "ELU 1",
            "ELU 2",
            "ELU 3",
            "ELU 4",
        ],
        "Nx": ["kN", 0, 0, 0, -301, -150, 55.5, 282, -4.5, -240, -163, -17, 332],
        "Vz": ["kN", 264, 138, 123, 152, 32.3, 163, 19, 88.15, 61.2, 29, 47, 21],
        "My": ["kNm", -172, -90, -81, -234, 143, -278, 159, -97, -38, 60, -39, 46.13],
    }
)

wall_summary = ShearWallSummary(conc, steel, sections, forces)

**Design reinforcement for all walls**

In [3]:
wall_summary.design()

✅ Design completed for every section of the summary.


,Level,Label,t,lw,hw,cc,dbh,sh,dbv,sv
0,,,cm,m,m,mm,mm,cm,mm,cm
1,Level 1,M1,20,3,3,25,10,30,10,30
2,Level 2,M1,20,3,3,25,10,30,10,30
3,Level 1,M2,20,2,3,25,10,30,10,30


**Check walls after design**

In [4]:
wall_summary.check()

,Level,Label,t,lw,hw,Horiz. (each face),Vert. (each face),ρt,ρl,Comb.,Vu,Nu,ØVn,DCR,Warnings,Status
0,,,cm,m,m,,,,,,kN,kN,kN,,,
1,Level 1,M1,20.0,3.0,3.0,Ø10/30,Ø10/30,0.00262,0.00262,ELU 1,264.0,0.0,1057.3,0.25,-,✅
2,Level 2,M1,20.0,3.0,3.0,Ø10/30,Ø10/30,0.00262,0.00262,ELU 2,163.0,55.5,1057.3,0.154,-,✅
3,Level 1,M2,20.0,2.0,3.0,Ø10/30,Ø10/30,0.00262,0.00262,ELU 1,61.2,-240.0,540.5,0.113,-,✅


**Export designed rebar to Excel**

In [5]:
wall_summary.export_design("Shear_wall_design.xlsx")

✅ Sections and forces written to Shear_wall_design.xlsx


**Import edited design from Excel and re-check**

In [6]:
wall_summary.import_design("Shear_wall_design.xlsx")
wall_summary.check()

✅ Sections and forces read from Shear_wall_design.xlsx


,Level,Label,t,lw,hw,Horiz. (each face),Vert. (each face),ρt,ρl,Comb.,Vu,Nu,ØVn,DCR,Warnings,Status
0,,,cm,m,m,,,,,,kN,kN,kN,,,
1,Level 1,M1,20.0,3.0,3.0,Ø10/30,Ø10/30,0.00262,0.00262,ELU 1,264.0,0.0,1057.3,0.25,-,✅
2,Level 2,M1,20.0,3.0,3.0,Ø10/30,Ø10/30,0.00262,0.00262,ELU 2,163.0,55.5,1057.3,0.154,-,✅
3,Level 1,M2,20.0,2.0,3.0,Ø10/30,Ø10/30,0.00262,0.00262,ELU 1,61.2,-240.0,540.5,0.113,-,✅


**View detailed shear results**

In [7]:
wall_summary.shear_results()

,Label,Comb.,"ρt,min","ρt,req",ρt,"ρl,min",ρl,Vu,ØVc,ØVs,ØVn,"ØVn,max","Vu≤ØVn,max",Vu≤ØVn,DCR
0,,,,,,,,kN,kN,kN,kN,kN,,,
1,M1,ELU 1,0.0025,0.0025,0.00262,0.0025,0.00262,264.0,562.5,494.8,1057.3,1485.0,True,True,0.25
2,M1,ELU 2,0.0025,0.0025,0.00262,0.0025,0.00262,138.0,562.5,494.8,1057.3,1485.0,True,True,0.131
3,M1,ELU 3,0.0025,0.0025,0.00262,0.0025,0.00262,123.0,562.5,494.8,1057.3,1485.0,True,True,0.116
4,M1,ELU 4,0.0025,0.0025,0.00262,0.0025,0.00262,152.0,326.88,494.8,821.68,1485.0,True,True,0.185
5,M1,ELU 1,0.0025,0.0025,0.00262,0.0025,0.00262,32.3,354.78,494.8,849.58,1485.0,True,True,0.038
6,M1,ELU 2,0.0025,0.0025,0.00262,0.0025,0.00262,163.0,562.5,494.8,1057.3,1485.0,True,True,0.154
7,M1,ELU 3,0.0025,0.0025,0.00262,0.0025,0.00262,19.0,562.5,494.8,1057.3,1485.0,True,True,0.018
8,M1,ELU 4,0.0025,0.0025,0.00262,0.0025,0.00262,88.15,381.67,494.8,876.47,1485.0,True,True,0.101
9,M2,ELU 1,0.0025,0.0025,0.00262,0.0025,0.00262,61.2,210.65,329.87,540.52,990.0,True,True,0.113


**Export detailed results to Word**

In [8]:
wall_summary.results_detailed_doc(index=1)

✅ Results exported to Shear_Wall_Summary_ACI 318-19.docx
